# 19 — Human-in-the-Loop Approval Workflow

Our Investment AI system can identify risk and recommend an action.

But for a consequential decision, the AI should not automatically execute the action.

In this notebook:

```text
Portfolio risk detected
        ↓
AI recommends action
        ↓
Approval request created
        ↓
PENDING_APPROVAL
        ↓
Human reviewer
   ├── APPROVE
   └── REJECT
        ↓
Only approved actions can proceed
```

We will reuse Lakebase as the durable approval store.


## 1. Install dependencies


In [0]:
# NOTE: install pure-Python psycopg, NOT psycopg[binary].
# The binary wheel bundles its own OpenSSL, which fails the FIPS self-test
# on this compute (SIGABRT at import). Pure psycopg loads the FIPS-compliant
# system libpq via ctypes instead.
%pip uninstall -q -y psycopg-binary
%pip install -q --upgrade "psycopg>=3.2" "databricks-sdk>=0.118.0"
dbutils.library.restartPython()


## 2. Connect to the existing Lakebase project

We reuse the same Lakebase project from the state and memory notebook.

```text
Project  : investment-ai-agent-state
Branch   : production
Endpoint : primary
Database : databricks_postgres
```

The notebook generates a short-lived database credential programmatically.


In [0]:
import uuid
from datetime import datetime, timezone

import psycopg
from databricks.sdk import WorkspaceClient

PROJECT = "investment-ai-agent-state"
BRANCH = "production"
ENDPOINT = "primary"
DATABASE = "databricks_postgres"
SCHEMA = "agent_state"

ENDPOINT_NAME = f"projects/{PROJECT}/branches/{BRANCH}/endpoints/{ENDPOINT}"

w = WorkspaceClient()

endpoint = w.postgres.get_endpoint(name=ENDPOINT_NAME)
current_user = w.current_user.me()
credential = w.postgres.generate_database_credential(endpoint=ENDPOINT_NAME)

DB_HOST = endpoint.status.hosts.host
DB_USER = current_user.user_name
DB_PASSWORD = credential.token

def get_conn():
    return psycopg.connect(
        host=DB_HOST,
        dbname=DATABASE,
        user=DB_USER,
        password=DB_PASSWORD,
        sslmode="require",
    )

print("Connected to Lakebase endpoint:", ENDPOINT)


## 3. Create the human approval table

This table gives us an audit trail of what the AI recommended, why it recommended it, the approval status, who reviewed it, and when.


In [0]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

        cur.execute(f'''
            CREATE TABLE IF NOT EXISTS {SCHEMA}.human_approvals (
                approval_id TEXT PRIMARY KEY,
                session_id TEXT,
                portfolio_name TEXT NOT NULL,
                recommended_action TEXT NOT NULL,
                reason TEXT NOT NULL,
                risk_status TEXT NOT NULL,
                status TEXT NOT NULL,
                requested_at TIMESTAMPTZ NOT NULL,
                reviewed_at TIMESTAMPTZ,
                reviewed_by TEXT,
                reviewer_comment TEXT
            )
        ''')

    conn.commit()

print("Table ready:", f"{SCHEMA}.human_approvals")


## 4. Use a familiar portfolio-risk scenario

We reuse the same concentration example from the previous notebooks.

```text
Position weight : 39.34%
Policy threshold: 10%
Risk status     : BREACH
```

The AI may recommend action, but it should not execute anything automatically.


In [0]:
portfolio_result = {
    "session_id": "hitl-demo-session-01",
    "portfolio_name": "Synthetic Growth Fund 01",
    "largest_holding": "Synthetic Compute Company 01",
    "position_weight": 39.34,
    "threshold": 10.00,
    "risk_status": "BREACH",
}

recommended_action = (
    "Escalate the concentration breach for portfolio-manager review "
    "and prepare a remediation recommendation."
)

reason = (
    f"{portfolio_result['largest_holding']} represents "
    f"{portfolio_result['position_weight']}% of the portfolio, "
    f"which exceeds the {portfolio_result['threshold']}% concentration threshold."
)

print("Risk status:", portfolio_result["risk_status"])
print("Recommended action:", recommended_action)


## 5. Create a pending approval request

This is the key control point.

The system records the recommendation but deliberately stops at:

`PENDING_APPROVAL`


In [0]:
approval_id = str(uuid.uuid4())

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f'''
            INSERT INTO {SCHEMA}.human_approvals (
                approval_id,
                session_id,
                portfolio_name,
                recommended_action,
                reason,
                risk_status,
                status,
                requested_at
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            ''',
            (
                approval_id,
                portfolio_result["session_id"],
                portfolio_result["portfolio_name"],
                recommended_action,
                reason,
                portfolio_result["risk_status"],
                "PENDING_APPROVAL",
                datetime.now(timezone.utc),
            ),
        )
    conn.commit()

print("Approval request created")
print("Approval ID:", approval_id)
print("Status: PENDING_APPROVAL")


## 6. Inspect the pending request

At this point, the workflow must stop.

No trade, rebalance, or consequential action is allowed to proceed yet.


In [0]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT
                approval_id,
                portfolio_name,
                recommended_action,
                risk_status,
                status,
                requested_at
            FROM {SCHEMA}.human_approvals
            WHERE approval_id = %s
            ''',
            (approval_id,),
        )
        row = cur.fetchone()

print(row)


## 7. Human review step

For the POC, we simulate a portfolio manager reviewing the recommendation.

Choose one:

```python
HUMAN_DECISION = "APPROVED"
```

or:

```python
HUMAN_DECISION = "REJECTED"
```


In [0]:
HUMAN_DECISION = "APPROVED"
REVIEWED_BY = "portfolio.manager@example.com"
REVIEWER_COMMENT = (
    "Approve escalation for immediate review. "
    "Do not execute any trade automatically."
)

assert HUMAN_DECISION in {"APPROVED", "REJECTED"}

print("Human decision:", HUMAN_DECISION)


## 8. Persist the human decision


In [0]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f'''
            UPDATE {SCHEMA}.human_approvals
            SET
                status = %s,
                reviewed_at = %s,
                reviewed_by = %s,
                reviewer_comment = %s
            WHERE approval_id = %s
            ''',
            (
                HUMAN_DECISION,
                datetime.now(timezone.utc),
                REVIEWED_BY,
                REVIEWER_COMMENT,
                approval_id,
            ),
        )
    conn.commit()

print("Approval updated:", HUMAN_DECISION)


## 9. Controlled execution gate

The rule is simple:

```text
PENDING_APPROVAL → STOP
REJECTED         → STOP
APPROVED         → CONTINUE
```

Even after approval, this POC does not place a real trade. It only demonstrates that the workflow is allowed to move to the next controlled step.


In [0]:
def execute_if_approved(approval_id):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute(
                f'''
                SELECT
                    status,
                    portfolio_name,
                    recommended_action,
                    reviewed_by
                FROM {SCHEMA}.human_approvals
                WHERE approval_id = %s
                ''',
                (approval_id,),
            )
            row = cur.fetchone()

    if row is None:
        return {
            "execution_status": "BLOCKED",
            "reason": "Approval record not found",
        }

    status, portfolio_name, action, reviewed_by = row

    if status != "APPROVED":
        return {
            "execution_status": "BLOCKED",
            "reason": f"Human approval required. Current status: {status}",
        }

    return {
        "execution_status": "READY_FOR_CONTROLLED_EXECUTION",
        "portfolio_name": portfolio_name,
        "approved_action": action,
        "approved_by": reviewed_by,
    }

execution_result = execute_if_approved(approval_id)
execution_result


## 10. Verify the complete audit record


In [0]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f'''
            SELECT
                approval_id,
                portfolio_name,
                risk_status,
                status,
                requested_at,
                reviewed_at,
                reviewed_by,
                reviewer_comment
            FROM {SCHEMA}.human_approvals
            WHERE approval_id = %s
            ''',
            (approval_id,),
        )
        final_record = cur.fetchone()

print(final_record)


# What this demonstrates

The AI is allowed to:

```text
✓ detect the breach
✓ explain the risk
✓ recommend an action
✓ create an approval request
```

But the AI is not allowed to:

```text
✗ approve its own recommendation
✗ bypass the human reviewer
✗ execute a consequential action before approval
```

That separation is the core Human-in-the-Loop pattern.


# ✅ Human-in-the-Loop complete

Our production pattern is now:

```text
Agent / Risk Workflow
        ↓
Governed BREACH result
        ↓
AI recommendation
        ↓
Lakebase approval record
        ↓
PENDING_APPROVAL
        ↓
Human reviewer
   ├── APPROVED
   └── REJECTED
        ↓
Execution gate
        ↓
Only approved workflow can continue
```

The key principle is:

> **AI can recommend. Humans retain authority over consequential actions.**

### Next

**Production Packaging & Deployment**

We will stop treating the system as a collection of notebooks and package the multi-agent workflow behind one production entry point.
